### Импорты

In [1]:
import pandas as pd
import numpy as np

from datasets import load_dataset

import nltk
from nltk import word_tokenize, pos_tag
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import f1_score, make_scorer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier


c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sklearnex import patch_sklearn
patch_sklearn()

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


### NLTK

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

lemmatizer = WordNetLemmatizer()


[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading punkt_tab: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading averaged_perceptron_tagger: <urlopen error
[nltk_data]     [Errno 11001] getaddrinfo failed>
[nltk_data] Error loading wordnet: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading averaged_perceptron_tagger_eng: <urlopen
[nltk_data]     error [Errno 11001] getaddrinfo failed>


### Функции предобработки

In [4]:
def preprocess_raw(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    return " ".join(tokens)

def preprocess_lemma_nj(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    tagged = pos_tag(tokens)
    filtered = [word for word, tag in tagged if tag.startswith("N") or tag.startswith("J")]
    lemmas = [lemmatizer.lemmatize(t) for t in filtered]
    return " ".join(lemmas)


### Параметры кросс-валидации

In [5]:
scoring = {
    "f1_micro": make_scorer(f1_score, average="micro"),
    "f1_macro": make_scorer(f1_score, average="macro"),
    "f1_weighted": make_scorer(f1_score, average="weighted")
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run_cv(model, X, y, dataset_name, representation_name):
    cv_results = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False
    )

    fold_df = pd.DataFrame({
        "dataset": dataset_name,
        "representation": representation_name,
        "model": type(model).__name__,
        "fold": [1, 2, 3, 4, 5],
        "f1_micro": cv_results["test_f1_micro"],
        "f1_macro": cv_results["test_f1_macro"],
        "f1_weighted": cv_results["test_f1_weighted"]
    })

    summary_df = pd.DataFrame([{
        "dataset": dataset_name,
        "representation": representation_name,
        "model": type(model).__name__,
        "cv_f1_micro_mean": fold_df["f1_micro"].mean(),
        "cv_f1_micro_std": fold_df["f1_micro"].std(),
        "cv_f1_macro_mean": fold_df["f1_macro"].mean(),
        "cv_f1_macro_std": fold_df["f1_macro"].std(),
        "cv_f1_weighted_mean": fold_df["f1_weighted"].mean(),
        "cv_f1_weighted_std": fold_df["f1_weighted"].std()
    }])

    return fold_df, summary_df


### Emotion: загрузка датасета

In [6]:
dataset_emotion = load_dataset("emotion")

train_texts = list(dataset_emotion["train"]["text"])
train_labels = np.array(dataset_emotion["train"]["label"])

test_texts = list(dataset_emotion["test"]["text"])
test_labels = np.array(dataset_emotion["test"]["label"])

print("emotion train:", len(train_texts))
print("emotion test:", len(test_texts))


'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/emotion/resolve/cab853a1dbdf4c42c2b3ef2173804746df8825fe/emotion.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since emotion couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'split' at C:\Users\Masha\.cache\huggingface\datasets\emotion\split\0.0.0\cab853a1dbdf4c42c2b3ef2173804746df8825fe (last modified on Wed Feb 18 11:25:49 2026).


emotion train: 16000
emotion test: 2000


### Emotion: подготовка текстов и векторизация

In [7]:
train_raw = [preprocess_raw(t) for t in train_texts]

vec_emotion_count = CountVectorizer()
X_train_emotion_count = vec_emotion_count.fit_transform(train_raw)

print("X_train_emotion_count:", X_train_emotion_count.shape)


X_train_emotion_count: (16000, 15184)


### Emotion: модели

In [8]:
emotion_models = {
    "RandomForestClassifier": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        max_features="sqrt",
        min_samples_leaf=2,
        min_samples_split=4,
        random_state=42,
        n_jobs=-1
    ),
    "DecisionTreeClassifier": DecisionTreeClassifier(
        max_depth=None,
        random_state=42
    ),
    "GradientBoostingClassifier": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ),
    "AdaBoostClassifier": AdaBoostClassifier(
        n_estimators=100,
        learning_rate=0.5,
        random_state=42
    )
}

### Emotion: кросс-валидация

In [9]:
emotion_cv_rows = []

for model_name, model in emotion_models.items():
    cv_results = cross_validate(
        model,
        X_train_emotion_count,
        train_labels,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    for fold_idx in range(5):
        emotion_cv_rows.append({
            "dataset": "emotion",
            "representation": "raw_count",
            "model": model_name,
            "fold": fold_idx + 1,
            "f1_micro": cv_results["test_f1_micro"][fold_idx],
            "f1_macro": cv_results["test_f1_macro"][fold_idx],
            "f1_weighted": cv_results["test_f1_weighted"][fold_idx]
        })

df_emotion_folds = pd.DataFrame(emotion_cv_rows)
print("Emotion: результаты по фолдам")
display(df_emotion_folds)

df_emotion_summary = (
    df_emotion_folds
    .groupby(["dataset", "representation", "model"], as_index=False)
    .agg(
        cv_f1_micro_mean=("f1_micro", "mean"),
        cv_f1_micro_std=("f1_micro", "std"),
        cv_f1_macro_mean=("f1_macro", "mean"),
        cv_f1_macro_std=("f1_macro", "std"),
        cv_f1_weighted_mean=("f1_weighted", "mean"),
        cv_f1_weighted_std=("f1_weighted", "std"),
    )
    .sort_values("cv_f1_micro_mean", ascending=False)
)

print("\nEmotion: средние значения и стандартные отклонения")
display(df_emotion_summary)

Emotion: результаты по фолдам


,dataset,representation,model,fold,f1_micro,f1_macro,f1_weighted
0,emotion,raw_count,RandomForestClassifier,1,0.869687,0.825272,0.866736
1,emotion,raw_count,RandomForestClassifier,2,0.861875,0.816546,0.858843
2,emotion,raw_count,RandomForestClassifier,3,0.875000,0.844435,0.873170
3,emotion,raw_count,RandomForestClassifier,4,0.871875,0.828176,0.869029
4,emotion,raw_count,RandomForestClassifier,5,0.875000,0.837984,0.872853
5,emotion,raw_count,DecisionTreeClassifier,1,0.868750,0.828347,0.869365
6,emotion,raw_count,DecisionTreeClassifier,2,0.855625,0.818564,0.855741
7,emotion,raw_count,DecisionTreeClassifier,3,0.875938,0.847418,0.876024
8,emotion,raw_count,DecisionTreeClassifier,4,0.865313,0.829937,0.865394
9,emotion,raw_count,DecisionTreeClassifier,5,0.845312,0.819534,0.845787



Emotion: средние значения и стандартные отклонения


,dataset,representation,model,cv_f1_micro_mean,cv_f1_micro_std,cv_f1_macro_mean,cv_f1_macro_std,cv_f1_weighted_mean,cv_f1_weighted_std
2,emotion,raw_count,GradientBoostingClassifier,0.885875,0.003332,0.852079,0.007219,0.886185,0.003162
3,emotion,raw_count,RandomForestClassifier,0.870688,0.005414,0.830483,0.010927,0.868126,0.005846
1,emotion,raw_count,DecisionTreeClassifier,0.862187,0.011934,0.828760,0.011605,0.862462,0.011868
0,emotion,raw_count,AdaBoostClassifier,0.356875,0.004926,0.177150,0.018954,0.217332,0.005202


### 20_newsgroups(4): загрузка датасета

In [10]:
dataset_news = load_dataset("SetFit/20_newsgroups")

categories = [
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.graphics",
    "comp.windows.x"
]

train_news = dataset_news["train"].filter(lambda example: example["label_text"] in categories)
test_news = dataset_news["test"].filter(lambda example: example["label_text"] in categories)

train_news_texts = list(train_news["text"])
train_news_labels = np.array(train_news["label"])

test_news_texts = list(test_news["text"])
test_news_labels = np.array(test_news["label"])

print("20_newsgroups(4) train:", len(train_news_texts))
print("20_newsgroups(4) test:", len(test_news_texts))


'[WinError 10060] Попытка установить соединение была безуспешной, т.к. от другого компьютера за требуемое время не получен нужный отклик, или было разорвано уже установленное соединение из-за неверного отклика уже подключенного компьютера' thrown while requesting HEAD https://huggingface.co/datasets/SetFit/20_newsgroups/resolve/f1b91292074e7cfb69be58b642d583ec262f30ed/20_newsgroups.py
Retrying in 1s [Retry 1/5].
'[WinError 10060] Попытка установить соединение была безуспешной, т.к. от другого компьютера за требуемое время не получен нужный отклик, или было разорвано уже установленное соединение из-за неверного отклика уже подключенного компьютера' thrown while requesting HEAD https://huggingface.co/datasets/SetFit/20_newsgroups/resolve/f1b91292074e7cfb69be58b642d583ec262f30ed/20_newsgroups.py
Retrying in 2s [Retry 2/5].
'[WinError 10060] Попытка установить соединение была безуспешной, т.к. от другого компьютера за требуемое время не получен нужный отклик, или было разорвано уже установ

20_newsgroups(4) train: 2345
20_newsgroups(4) test: 1561


### 20_newsgroups(4): подготовка текстов и векторизация

In [11]:
train_news_nj = [preprocess_lemma_nj(t) for t in train_news_texts]

vec_news_tfidf = TfidfVectorizer()
X_train_news_nj_tfidf = vec_news_tfidf.fit_transform(train_news_nj)

print("X_train_news_nj_tfidf:", X_train_news_nj_tfidf.shape)


X_train_news_nj_tfidf: (2345, 12370)


### 20_newsgroups(4): модели

In [12]:
news_models = {
    "RandomForestClassifier": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        max_features="sqrt",
        min_samples_leaf=2,
        min_samples_split=4,
        random_state=42,
        n_jobs=-1
    ),
    "DecisionTreeClassifier": DecisionTreeClassifier(
        max_depth=None,
        random_state=42
    ),
    "GradientBoostingClassifier": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ),
    "AdaBoostClassifier": AdaBoostClassifier(
        n_estimators=50,
        learning_rate=0.5,
        random_state=42
    )
}

### 20_newsgroups(4): кросс-валидация

In [13]:
news_cv_rows = []

for model_name, model in news_models.items():
    cv_results = cross_validate(
        model,
        X_train_news_nj_tfidf,
        train_news_labels,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    for fold_idx in range(5):
        news_cv_rows.append({
            "dataset": "20_newsgroups(4)",
            "representation": "lemma_N+J_tfidf",
            "model": model_name,
            "fold": fold_idx + 1,
            "f1_micro": cv_results["test_f1_micro"][fold_idx],
            "f1_macro": cv_results["test_f1_macro"][fold_idx],
            "f1_weighted": cv_results["test_f1_weighted"][fold_idx]
        })

df_news_folds = pd.DataFrame(news_cv_rows)
print("20_newsgroups(4): результаты по фолдам")
display(df_news_folds)

df_news_summary = (
    df_news_folds
    .groupby(["dataset", "representation", "model"], as_index=False)
    .agg(
        cv_f1_micro_mean=("f1_micro", "mean"),
        cv_f1_micro_std=("f1_micro", "std"),
        cv_f1_macro_mean=("f1_macro", "mean"),
        cv_f1_macro_std=("f1_macro", "std"),
        cv_f1_weighted_mean=("f1_weighted", "mean"),
        cv_f1_weighted_std=("f1_weighted", "std"),
    )
    .sort_values("cv_f1_micro_mean", ascending=False)
)

print("\n20_newsgroups(4): средние значения и стандартные отклонения")
display(df_news_summary)

20_newsgroups(4): результаты по фолдам


,dataset,representation,model,fold,f1_micro,f1_macro,f1_weighted
0,20_newsgroups(4),lemma_N+J_tfidf,RandomForestClassifier,1,0.744136,0.743147,0.743414
1,20_newsgroups(4),lemma_N+J_tfidf,RandomForestClassifier,2,0.782516,0.783907,0.783977
2,20_newsgroups(4),lemma_N+J_tfidf,RandomForestClassifier,3,0.727079,0.726313,0.726833
3,20_newsgroups(4),lemma_N+J_tfidf,RandomForestClassifier,4,0.733475,0.736177,0.736192
4,20_newsgroups(4),lemma_N+J_tfidf,RandomForestClassifier,5,0.765458,0.765389,0.765794
5,20_newsgroups(4),lemma_N+J_tfidf,DecisionTreeClassifier,1,0.628998,0.627288,0.627416
6,20_newsgroups(4),lemma_N+J_tfidf,DecisionTreeClassifier,2,0.603412,0.600953,0.601086
7,20_newsgroups(4),lemma_N+J_tfidf,DecisionTreeClassifier,3,0.556503,0.554652,0.554881
8,20_newsgroups(4),lemma_N+J_tfidf,DecisionTreeClassifier,4,0.618337,0.616987,0.617223
9,20_newsgroups(4),lemma_N+J_tfidf,DecisionTreeClassifier,5,0.599147,0.597576,0.598010



20_newsgroups(4): средние значения и стандартные отклонения


,dataset,representation,model,cv_f1_micro_mean,cv_f1_micro_std,cv_f1_macro_mean,cv_f1_macro_std,cv_f1_weighted_mean,cv_f1_weighted_std
3,20_newsgroups(4),lemma_N+J_tfidf,RandomForestClassifier,0.750533,0.023063,0.750987,0.023349,0.751242,0.023275
2,20_newsgroups(4),lemma_N+J_tfidf,GradientBoostingClassifier,0.712154,0.010001,0.716302,0.009584,0.716468,0.009429
0,20_newsgroups(4),lemma_N+J_tfidf,AdaBoostClassifier,0.630277,0.022945,0.637825,0.022363,0.638139,0.022372
1,20_newsgroups(4),lemma_N+J_tfidf,DecisionTreeClassifier,0.601279,0.027719,0.599491,0.027817,0.599723,0.027788


### Итоговая таблица

In [14]:
df_cv_all = pd.concat([df_emotion_summary, df_news_summary], ignore_index=True)
display(df_cv_all.sort_values(["dataset", "cv_f1_micro_mean"], ascending=[True, False]))

,dataset,representation,model,cv_f1_micro_mean,cv_f1_micro_std,cv_f1_macro_mean,cv_f1_macro_std,cv_f1_weighted_mean,cv_f1_weighted_std
4,20_newsgroups(4),lemma_N+J_tfidf,RandomForestClassifier,0.750533,0.023063,0.750987,0.023349,0.751242,0.023275
5,20_newsgroups(4),lemma_N+J_tfidf,GradientBoostingClassifier,0.712154,0.010001,0.716302,0.009584,0.716468,0.009429
6,20_newsgroups(4),lemma_N+J_tfidf,AdaBoostClassifier,0.630277,0.022945,0.637825,0.022363,0.638139,0.022372
7,20_newsgroups(4),lemma_N+J_tfidf,DecisionTreeClassifier,0.601279,0.027719,0.599491,0.027817,0.599723,0.027788
0,emotion,raw_count,GradientBoostingClassifier,0.885875,0.003332,0.852079,0.007219,0.886185,0.003162
1,emotion,raw_count,RandomForestClassifier,0.870688,0.005414,0.830483,0.010927,0.868126,0.005846
2,emotion,raw_count,DecisionTreeClassifier,0.862187,0.011934,0.828760,0.011605,0.862462,0.011868
3,emotion,raw_count,AdaBoostClassifier,0.356875,0.004926,0.177150,0.018954,0.217332,0.005202


### Вывод

In [15]:
print("Emotion:")
print(df_emotion_summary.sort_values("cv_f1_micro_mean", ascending=False))

print("\n20_newsgroups(4):")
print(df_news_summary.sort_values("cv_f1_micro_mean", ascending=False))


Emotion:
   dataset representation                       model  cv_f1_micro_mean  \
2  emotion      raw_count  GradientBoostingClassifier          0.885875   
3  emotion      raw_count      RandomForestClassifier          0.870688   
1  emotion      raw_count      DecisionTreeClassifier          0.862187   
0  emotion      raw_count          AdaBoostClassifier          0.356875   

   cv_f1_micro_std  cv_f1_macro_mean  cv_f1_macro_std  cv_f1_weighted_mean  \
2         0.003332          0.852079         0.007219             0.886185   
3         0.005414          0.830483         0.010927             0.868126   
1         0.011934          0.828760         0.011605             0.862462   
0         0.004926          0.177150         0.018954             0.217332   

   cv_f1_weighted_std  
2            0.003162  
3            0.005846  
1            0.011868  
0            0.005202  

20_newsgroups(4):
            dataset   representation                       model  \
3  20_newsgroups(